In [1]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = "C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

staging_table_name = "staging.Integration.employee_Staging"
wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
# spark.sql("SHOW CATALOGS").show(truncate=False)
# spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
# spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
# spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)



Namespace: dimension
+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|dimension|payment_method  |false      |
|dimension|supplier        |false      |
|dimension|city            |false      |
|dimension|stock_item      |false      |
|dimension|customer        |false      |
|dimension|date            |false      |
|dimension|transaction_type|false      |
|dimension|employee        |false      |
+---------+----------------+-----------+


Namespace: fact
+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|fact     |purchase     |false      |
|fact     |stock_holding|false      |
|fact     |order        |false      |
|fact     |movement     |false      |
|fact     |sale         |false      |
|fact     |transaction  |false      |
+---------+-------------+-----------+


Namespace: integration
+-----------+-----------------------+-----------+
|namespace  |


### Namespace: dimension

|namespace|tableName       |isTemporary|
|---------|----------------|-----------|
|dimension|payment_method  |false      |
|dimension|supplier        |false      |
|dimension|city            |false      |
|dimension|stock_item      |false      |
|dimension|customer        |false      |
|dimension|date            |false      |
|dimension|transaction_type|false      |
|dimension|employee        |false      |



### Namespace: fact

|namespace|tableName    |isTemporary|
|---------|-------------|-----------|
|fact     |purchase     |false      |
|fact     |stock_holding|false      |
|fact     |order        |false      |
|fact     |movement     |false      |
|fact     |sale         |false      |
|fact     |transaction  |false      |



### Namespace: integration

|namespace  |tableName              |isTemporary|
|-----------|-----------------------|-----------|
|integration|employee_staging       |false      |
|integration|transactiontype_staging|false      |
|integration|paymentmethod_staging  |false      |
|integration|supplier_staging       |false      |
|integration|movement_staging       |false      |
|integration|customer_staging       |false      |
|integration|city_staging           |false      |
|integration|stockholding_staging   |false      |
|integration|lineage                |false      |
|integration|order_staging          |false      |
|integration|stockitem_staging      |false      |
|integration|transaction_staging    |false      |
|integration|etl_cutoff             |false      |
|integration|sale_staging           |false      |
|integration|purchase_staging       |false      |



In [3]:
df_sales = spark.table("reporting.Fact.Sale").alias("sales")
df_customer = spark.table("reporting.Dimension.Customer").alias("customer")
# df_sales.printSchema()
df_customer.printSchema()


root
 |-- Customer Key: integer (nullable = true)
 |-- WWI Customer ID: integer (nullable = true)
 |-- Customer: string (nullable = true)
 |-- Bill To Customer: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Buying Group: string (nullable = true)
 |-- Primary Contact: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Valid From: timestamp (nullable = true)
 |-- Valid To: timestamp (nullable = true)
 |-- Lineage Key: integer (nullable = true)



In [ ]:

df_customer_sales = df_customer.join(
    df_sales, sf.col("customer.Customer Key") == sf.col("sales.Customer Key"), "inner"
)
df_customer_sales = df_customer_sales.select(
    sf.col("customer.Customer Key").alias("CustomerKey"),
    sf.col("customer.Customer Name").alias("CustomerName"),
    sf.col("sales.Sale Key").alias("SaleKey"),
    sf.col("sales.Sale Amount").alias("SaleAmount"),
    sf.col("sales.Sale Date").alias("SaleDate")
)

df_customer_sales.show(5, truncate=False)

{"ts": "2025-12-06 12:43:22.800", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[INVALID_EXTRACT_BASE_FIELD_TYPE] Can't extract a value from \"customer\". Need a complex type [STRUCT, ARRAY, MAP] but got \"STRING\". SQLSTATE: 42000", "context": {"file": "line 11 in cell [2]", "line": "", "fragment": "col", "errorClass": "INVALID_EXTRACT_BASE_FIELD_TYPE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o75.select.\n: org.apache.spark.sql.AnalysisException: [INVALID_EXTRACT_BASE_FIELD_TYPE] Can't extract a value from \"customer\". Need a complex type [STRUCT, ARRAY, MAP] but got \"STRING\". SQLSTATE: 42000\r\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.dataTypeUnsupportedByExtractValueError(QueryCompilationErrors.scala:1418)\r\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:71)\r\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.$anonfun$resolveCandidates$

AnalysisException: [INVALID_EXTRACT_BASE_FIELD_TYPE] Can't extract a value from "customer". Need a complex type [STRUCT, ARRAY, MAP] but got "STRING". SQLSTATE: 42000